In [3]:
from pathlib import Path

src_path = Path('sample/pcfg3_10k.txt')
lines = src_path.read_text(encoding='utf-8').splitlines()

# PCFG symbol -> vocabulary mapping
symbol_to_words = {
    'A':{'a'},
    'B':{'b'},
    'C':{'c'},
    'D':{'d'},
    'E':{'e'},
    'F':{'f'},
    'G':{'g'},
    'H':{'h'},
    'I':{'i'},
    'J':{'j'},
    'K':{'k'},
    'L':{'l'},
    'M':{'m'},
    'N':{'n'},
    'O':{'o'},
    'P':{'p'},
    'Q':{'q'},
    'R':{'r'},
    'S':{'s'},
    'T':{'t'},
    'U':{'u'},
    'V':{'v'},
    'W':{'w'},
    'X':{'x'},
    'Y':{'y'},
    'Z':{'z'}
}
wordCounts = 0
for key in symbol_to_words:
    wordCounts += len(symbol_to_words[key])
print(wordCounts)
word_to_symbol = {}
for symbol, words in symbol_to_words.items():
    for w in words:
        if w in word_to_symbol:
            raise ValueError(f'Duplicate mapping for word: {w}')
        word_to_symbol[w] = symbol

vocab = sorted({tok for line in lines for tok in line.split()})
missing = sorted(set(vocab) - set(word_to_symbol))
if missing:
    raise ValueError(f'Missing symbol mapping for: {missing}')

symbol_lines = [' '.join(word_to_symbol[tok] for tok in line.split()) for line in lines]

out_path = Path('sample/pcfg3_10k_symbols.txt')
out_path.write_text('\n'.join(symbol_lines) + '\n', encoding='utf-8')


26


613656

In [41]:
import nltk
from nltk.grammar import PCFG
from nltk.parse import ViterbiParser
from collections import Counter

# --- Config ---
corpus_filepath = './sample/pcfg2_50k.txt'
output_csv = 'pcfg2.csv'
num_sentences = 50000

# --- Build grammar string ---
dummy_grammar_str = """
    S -> NP VP [0.5]
    S -> NP V [0.5]
    PP -> IN NP [1]
    NP -> DT N [0.333333333333]
    NP -> ADJP NP [0.333333333333]
    NP -> NP PP [0.333333333333]
    
    ADJP -> ADJP PP [0.5]
    ADJP -> DEG JJ [0.5]
    VP -> VP PP [0.2]
    VP -> V NP [0.2]
    VP -> V PP [0.2]
    VP -> MD VP [0.2]
    VP -> VP RB [0.2]
"""

for symbol, words in symbol_to_words.items():
    prob = 1.0 / len(words)
    for word in words:
        dummy_grammar_str += f"\n    {symbol} -> '{word}' [{prob}]"

grammar = PCFG.fromstring(dummy_grammar_str)
parser = ViterbiParser(grammar)

# --- Parse corpus ---
production_counts = Counter()
lhs_counts = Counter()

with open(corpus_filepath, 'r', encoding='utf-8') as f:
    sentences = [line.strip().split() for line in f.readlines()][:num_sentences]

for i, tokens in enumerate(sentences):
    if i % 500 == 0:
        print(f"Parsed {i}/{len(sentences)} sentences...")
    try:
        trees = list(parser.parse(tokens))
        if trees:
            tree = trees[0]
            for prod in tree.productions():
                production_counts[prod] += 1
                lhs_counts[prod.lhs()] += 1
    except ValueError:
        continue

print(f"Done parsing. {len(production_counts)} unique productions found.")

Parsed 0/49998 sentences...
Parsed 500/49998 sentences...
Parsed 1000/49998 sentences...
Parsed 1500/49998 sentences...
Parsed 2000/49998 sentences...
Parsed 2500/49998 sentences...
Parsed 3000/49998 sentences...
Parsed 3500/49998 sentences...
Parsed 4000/49998 sentences...
Parsed 4500/49998 sentences...
Parsed 5000/49998 sentences...
Parsed 5500/49998 sentences...
Parsed 6000/49998 sentences...
Parsed 6500/49998 sentences...
Parsed 7000/49998 sentences...
Parsed 7500/49998 sentences...
Parsed 8000/49998 sentences...
Parsed 8500/49998 sentences...
Parsed 9000/49998 sentences...
Parsed 9500/49998 sentences...
Parsed 10000/49998 sentences...
Parsed 10500/49998 sentences...
Parsed 11000/49998 sentences...
Parsed 11500/49998 sentences...
Parsed 12000/49998 sentences...
Parsed 12500/49998 sentences...
Parsed 13000/49998 sentences...
Parsed 13500/49998 sentences...
Parsed 14000/49998 sentences...
Parsed 14500/49998 sentences...
Parsed 15000/49998 sentences...
Parsed 15500/49998 sentences...


In [42]:
# --- Debug: inspect production counts ---

# Show all nonterminal productions and their counts
print("=== Nonterminal Production Counts ===")
for prod, count in sorted(production_counts.items(), key=lambda x: -x[1]):
    if not any(nltk.grammar.is_terminal(sym) for sym in prod.rhs()):
        print(f"  {prod}    count={count}")

print(f"\n=== LHS totals ===")
for lhs, total in sorted(lhs_counts.items(), key=lambda x: str(x[0])):
    print(f"  {lhs}: {total}")

# Show top 20 preterminal (word) productions
print("\n=== Top 20 Preterminal Productions ===")
preterminals = [(prod, count) for prod, count in production_counts.items()
                if any(nltk.grammar.is_terminal(sym) for sym in prod.rhs())]
for prod, count in sorted(preterminals, key=lambda x: -x[1])[:20]:
    lhs = prod.lhs()
    prob = count / lhs_counts[lhs]
    print(f"  {prod}    count={count}  P={prob:.4f}")

# Show a single parsed tree as an example
print("\n=== Example Parse Tree ===")
example = sentences[2]
print(f"Sentence: {' '.join(example)}")
trees = list(parser.parse(example))
if trees:
    tree = trees[0]
    print(tree)
    print("\nProductions from this tree:")
    for prod in tree.productions():
        is_term = any(nltk.grammar.is_terminal(sym) for sym in prod.rhs())
        label = "preterminal" if is_term else "nonterminal"
        print(f"  [{label}] {prod}")
else:
    print("No parse found.")

=== Nonterminal Production Counts ===
  NP -> DT N    count=100959
  PP -> IN NP    count=32481
  S -> NP VP    count=27901
  NP -> NP PP    count=25508
  VP -> V NP    count=20930
  S -> NP V    count=19647
  VP -> VP RB    count=8807
  VP -> V PP    count=6971
  NP -> ADJP NP    count=6262
  ADJP -> DEG JJ    count=6262
  VP -> MD VP    count=1991
  VP -> VP PP    count=2

=== LHS totals ===
  ADJP: 6262
  DEG: 6262
  DT: 100959
  IN: 32481
  JJ: 6262
  MD: 1991
  N: 100959
  NP: 132729
  PP: 32481
  RB: 8807
  S: 47548
  V: 47548
  VP: 38701

=== Top 20 Preterminal Productions ===
  DT -> 'the'    count=41543  P=0.4115
  DT -> 'a'    count=27404  P=0.2714
  DT -> 'this'    count=13613  P=0.1348
  IN -> 'in'    count=8112  P=0.2497
  IN -> 'on'    count=6904  P=0.2126
  IN -> 'at'    count=5940  P=0.1829
  N -> 'dogs'    count=5663  P=0.0561
  N -> 'cats'    count=5397  P=0.0535
  N -> 'students'    count=5157  P=0.0511
  N -> 'professors'    count=4886  P=0.0484
  IN -> 'with'    co

In [43]:
# --- Build rows from parsed productions ---
rows = []
seen_preterminals = set()

for prod, count in production_counts.items():
    lhs = str(prod.lhs())
    rhs = " ".join([str(sym) for sym in prod.rhs()])
    
    is_preterminal = any(nltk.grammar.is_terminal(sym) for sym in prod.rhs())
    lhs_type = "preterminal" if is_preterminal else "nonterminal"
    
    rhs_clean = rhs.replace("'", "")
    probability = count / lhs_counts[prod.lhs()]
    
    rows.append((lhs, lhs_type, rhs_clean, probability))
    
    if is_preterminal:
        seen_preterminals.add((lhs, rhs_clean))

# --- Add missing preterminal rules with probability 0 ---
for symbol, words in symbol_to_words.items():
    for word in words:
        if (symbol, word) not in seen_preterminals:
            rows.append((symbol, "preterminal", word, 0.0))

# --- Fix LHS groups that sum to 0: assign uniform probabilities ---
from collections import defaultdict

lhs_prob_sums = defaultdict(float)
lhs_indices = defaultdict(list)

for i, (lhs, lhs_type, rhs_clean, probability) in enumerate(rows):
    lhs_prob_sums[lhs] += probability
    lhs_indices[lhs].append(i)

for lhs, total in lhs_prob_sums.items():
    if abs(total) < 1e-9:
        n = len(lhs_indices[lhs])
        uniform_prob = 1.0 / n
        print(f"  {lhs}: sum=0, assigning uniform {uniform_prob:.6f} across {n} rules")
        for idx in lhs_indices[lhs]:
            lhs_val, lhs_type, rhs_clean, _ = rows[idx]
            rows[idx] = (lhs_val, lhs_type, rhs_clean, uniform_prob)

# --- Sort: nonterminal first, then preterminal; group by LHS ---
type_order = {"nonterminal": 0, "preterminal": 1}
rows.sort(key=lambda r: (type_order[r[1]], r[0], r[2]))

# --- Write to CSV ---
with open(output_csv, 'w', encoding='utf-8') as f:
    f.write("ID,LHS,LHS Type,RHS,Probability\n")
    for rule_id, (lhs, lhs_type, rhs_clean, probability) in enumerate(rows, start=1):
        f.write(f"{rule_id},{lhs},{lhs_type},{rhs_clean},{probability:.4f}\n")

print(f"Wrote {len(rows)} rules to {output_csv}")
print(f"  Nonterminal rules: {sum(1 for r in rows if r[1] == 'nonterminal')}")
print(f"  Preterminal rules: {sum(1 for r in rows if r[1] == 'preterminal')}")
print(f"  Missing words added with p=0: {sum(1 for r in rows if r[3] == 0.0)}")

Wrote 212 rules to pcfg2.csv
  Nonterminal rules: 12
  Preterminal rules: 200
  Missing words added with p=0: 0


In [45]:
import pandas as pd
import nltk
from nltk.grammar import PCFG, Nonterminal, ProbabilisticProduction
from nltk.parse import pchart
import math

def load_grammar_from_csv(csv_filepath):
    """Rebuilds the NLTK PCFG directly from your Kaggle submission file."""
    df = pd.read_csv(csv_filepath)
    rules = []
    
    for _, row in df.iterrows():
        lhs = Nonterminal(row['LHS'])
        
        # Determine if the RHS is a terminal word or two nonterminals
        if row['LHS Type'] == 'nonterminal':
            rhs = [Nonterminal(s) for s in str(row['RHS']).split()]
        else:
            rhs = [str(row['RHS'])]
            
        # Create the NLTK production rule
        rules.append(ProbabilisticProduction(lhs, rhs, prob=row['Probability']))
        
    start_sym = Nonterminal('S')
    return PCFG(start_sym, rules)

def evaluate_pcfg(csv_filepath, sample_filepath, num_test_sentences=100):
    print(f"Loading grammar from {csv_filepath}...")
    grammar = load_grammar_from_csv(csv_filepath)
    
    # Using InsideChartParser to get the total sum of all possible parse trees
    parser = pchart.InsideChartParser(grammar)
    
    with open(sample_filepath, 'r', encoding='utf-8') as f:
        sentences = [line.strip().split() for line in f.readlines()][:num_test_sentences]

    total_log_prob = 0.0
    parsed_count = 0
    failed_count = 0

    print(f"Evaluating on {num_test_sentences} sentences. This may take a moment...")
    
    for i, tokens in enumerate(sentences):
        try:
            # Parse the sentence
            parses = list(parser.parse(tokens))
            
            if parses:
                p_x = sum(tree.prob() for tree in parses)
                if p_x > 0:
                    total_log_prob += math.log2(p_x)
                    parsed_count += 1
                else:
                    failed_count += 1
                    
                    print(f"FAILED (0 prob): {' '.join(tokens)}")
            else:
                failed_count += 1
                
                print(f"FAILED (No parse): {' '.join(tokens)}")
                
        except ValueError:
            # Happens if a sentence contains a word not in your grammar
            failed_count += 1
            
        if (i + 1) % 20 == 0:
            print(f"Processed {i + 1}/{num_test_sentences}...")

    # Calculate metrics
    if parsed_count > 0:
        avg_log_prob = total_log_prob / parsed_count
        perplexity = 2 ** (-avg_log_prob)
    else:
        avg_log_prob = float('-inf')
        perplexity = float('inf')

    print("\n" + "="*40)
    print("EVALUATION RESULTS")
    print("="*40)
    print(f"Sentences successfully parsed: {parsed_count} / {num_test_sentences}")
    print(f"Sentences failed (0% prob):  {failed_count}")
    print(f"Average Log2 Likelihood:     {avg_log_prob:.4f} (Closer to 0 is better)")
    print(f"Perplexity:                  {perplexity:.2f} (Lower is better)")
    print("="*40)
    
    if failed_count > (num_test_sentences * 0.1):
        print("\nWARNING: High failure rate. Your grammar structure is missing rules required to parse the teacher's sentences.")

# --- Execute ---
# Provide your submission CSV and a fresh file of sampled sentences from the teacher
evaluate_pcfg('pcfg2.csv', './sample/pcfg2_50k.txt', num_test_sentences=1000)

Loading grammar from pcfg2.csv...
Evaluating on 10000 sentences. This may take a moment...
FAILED (No parse): very calm one machines at the projects might may chases the doctors home
Processed 20/10000...
Processed 40/10000...
FAILED (No parse): noisy at a professor the project in three professors on a city in a planets cried on the teachers upstairs today
Processed 60/10000...
Processed 80/10000...
FAILED (No parse): small at one libraries a dogs sneezed
Processed 100/10000...
Processed 120/10000...
FAILED (No parse): a music find the student inside home tomorrow
Processed 140/10000...
FAILED (No parse): this planet chase big in the songs a music
FAILED (No parse): five planets cries with the professors home
Processed 160/10000...
FAILED (No parse): bright on a teacher the project at seven artists sneezed
Processed 180/10000...
FAILED (No parse): a machine watched a professor in very slow the river home
Processed 200/10000...
Processed 220/10000...
FAILED (No parse): this songs solve 

KeyboardInterrupt: 